In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
office_table = dbutils.widgets.get("office_table")
client_table = dbutils.widgets.get("client_table")
payer_table = dbutils.widgets.get("payer_table")
billto_table = dbutils.widgets.get("billto_table")

In [0]:
display(
spark.sql(
f"""
CREATE OR REPLACE TEMP VIEW claim_src AS
SELECT
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(FacilityCode AS INT) AS FacilityCode,
  CAST(AcctNbr AS STRING) AS AcctNbr,
  NULL AS BillingProviderNPI,
  CAST(PCN AS STRING) AS PCN,
  CAST(ClaimDate AS DATE) AS ClaimDate,
  NULL AS STMTFFromThru,
  NULL AS BillType,
  NULL AS PayerClaimNumber,
  CAST(SourceSystemKey AS INT) AS SourceSystemKey,
  current_timestamp() AS _load_timestamp
FROM (
  WITH 
  claim_cte AS (
    SELECT
      to_date(CAST(d.date_entered_key AS STRING), 'yyyyMMdd') AS ReportingDate,
      ofc.OfficeNumber AS FacilityCode,
      d.invoice_number AS AcctNbr,
      REPLACE(d.PCN, '-', '') AS PCN,
      try_to_date(CAST(d.InitialBillDate AS STRING), 'yyyyMMdd') AS ClaimDate,
      0 AS SourceSystemKey
    FROM (
        SELECT
          a.office_key,
          a.invoice_number,
          a.client_key,
          a.InitialBillDate,
          a.payor_key,
          a.date_entered_key,  -- <-- Added missing comma here
          CASE
            WHEN UPPER(a.invoice_number) LIKE '%ADV%'
              THEN CONCAT('ADV - ', clt.SourceSystemId)
            ELSE CONCAT(clt.SourceSystemId, a.invoice_number)
          END AS PCN
        FROM (
            SELECT DISTINCT
              obd.client_key,
              CASE
                WHEN UPPER(obd.invoice_number) = 'ADV'
                  THEN CONCAT('ADV - ', clt.SourceSystemId)
                ELSE obd.invoice_number
              END AS invoice_number,
              obd.office_key,
              obd.invoice_date_key AS InitialBillDate,
              obd.date_entered_key,
              obd.payor_key
            FROM {source_table} obd
            LEFT JOIN {client_table} clt
              ON clt.ClientKey = obd.client_key
            WHERE obd.account_balance != 0
            AND obd.date_entered_key = date_format(DATE('{fetch_date}'), 'yyyyMMdd')
        ) a
        LEFT JOIN {client_table} clt
          ON clt.ClientKey = a.client_key
    ) d
    LEFT JOIN {office_table} ofc
      ON ofc.OfficeKey = d.office_key
    LEFT JOIN {payer_table} pd
      ON pd.PayerKey = d.payor_key
    LEFT JOIN {billto_table} bt
      ON bt.BillToId = pd.PayerID
  ),
  claim_clean AS (
    SELECT *,
    row_number() OVER (
      PARTITION BY AcctNbr, FacilityCode 
      ORDER BY AcctNbr
    ) AS rn
    FROM claim_cte
  )
  SELECT ReportingDate, FacilityCode, AcctNbr, PCN, ClaimDate, SourceSystemKey
  FROM claim_clean
  WHERE rn=1
) AS src
"""
)
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING claim_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 0

WHEN MATCHED THEN
UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.BillingProviderNPI = src.BillingProviderNPI,
    tgt.PCN = src.PCN,
    tgt.ClaimDate = src.ClaimDate,
    tgt.STMTFFromThru = src.STMTFFromThru,
    tgt.BillType = src.BillType,
    tgt.PayerClaimNumber = src.PayerClaimNumber,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    BillingProviderNPI,
    PCN,
    ClaimDate,
    STMTFFromThru,
    BillType,
    PayerClaimNumber,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.BillingProviderNPI,
    src.PCN,
    src.ClaimDate,
    src.STMTFFromThru,
    src.BillType,
    src.PayerClaimNumber,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)